<a href="https://colab.research.google.com/github/davidlealo/sic_ia_2026/blob/main/ejercicios_profesor/sic2026_ia_clase11_resenas_pln_PROFESOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Diccionario categorizado por distintos temas de ejemplo (Español a Inglés)
diccionario_temas = {
    "adjetivos": {
        "grande": "big",
        "pequeño": "small",
        "rápido": "fast",
        "lento": "slow"
    },
    "naturaleza": {
        "ola": "wave",
        "mar": "sea",
        "sol": "sun",
        "viento": "wind"
    },
    "comida": {
        "manzana": "apple",
        "pan": "bread",
        "agua": "water"
    },
    "tecnologia": {
        "pantalla": "screen",
        "reloj": "watch",
        "bateria": "battery"
    }
}

# Ejemplo de cómo acceder a los elementos
print("Traducción de 'grande':", diccionario_temas["adjetivos"]["grande"])
print("Traducción de 'ola':", diccionario_temas["naturaleza"]["ola"])

Traducción de 'grande': big
Traducción de 'ola': wave


In [ ]:
# Definimos un diccionario de traducción para nuestros términos específicos
diccionario_traduccion = {
    "granola": "mezcla de cereales tostados",
    "big wave": "gran ola"
}

def traducir_texto(texto, diccionario):
    # Convertimos a minúsculas para una comparación estandarizada
    texto_limpio = texto.lower()
    for termino_ingles, traduccion_espanol in diccionario.items():
        # Reemplazamos las ocurrencias en el texto
        texto_limpio = texto_limpio.replace(termino_ingles, traduccion_espanol)
    return texto_limpio

# Ejemplo de uso
ejemplo_resena = "Me comí un tazón de granola mientras miraba la película Big Wave."
resena_traducida = traducir_texto(ejemplo_resena, diccionario_traduccion)

print("Texto original: ", ejemplo_resena)
print("Texto traducido:", resena_traducida)

Texto original:  Me comí un tazón de granola mientras miraba la película Big Wave.
Texto traducido: me comí un tazón de mezcla de cereales tostados mientras miraba la película gran ola.


In [ ]:
"Texto traducido: me comí un tazón de mezcla de cereales tostados mientras miraba la"
"película"


In [ ]:
"Me comí un _______ de granola mientras miraba la película Big Wave"

"tazón"

In [ ]:
"Un pájaro vale más que mil"

In [ ]:
"Una imagen vale más que mil"

# Explicación línea a línea del notebook

## Instalación de dependencias

```python
!pip install sgmllib3k --use-pep517
```
El signo `!` indica que la línea se ejecuta como comando de terminal y no como código Python. Instala `sgmllib3k`, un parser de SGML que necesita `feedparser`. La opción `--use-pep517` obliga a pip a usar el sistema de construcción moderno de paquetes, lo que evita errores de instalación en algunos entornos (por ejemplo, Colab).

```python
!pip install feedparser
```
Instala `feedparser`, una librería para leer feeds RSS y Atom. **Nota:** ni `feedparser` ni `sgmllib3k` se usan en el resto del código, así que estas dos líneas se pueden eliminar sin afectar el resultado.

## 1. Obtención de noticias en vivo

```python
import requests
```
Importa `requests`, la librería estándar de facto para hacer peticiones HTTP (descargar contenido desde internet).

```python
import re
```
Importa `re`, el módulo de expresiones regulares de Python, que se usará para limpiar el texto.

```python
url = "https://api.spaceflightnewsapi.net/v4/articles/?limit=50"
```
Define la dirección de la API de Spaceflight News, un servicio público que entrega noticias sobre el espacio. El parámetro `?limit=50` pide 50 artículos.

```python
data = requests.get(url).json()
```
Hace dos cosas encadenadas:
1. `requests.get(url)` envía una petición GET a la API y recibe la respuesta.
2. `.json()` convierte el cuerpo de la respuesta (texto en formato JSON) en un diccionario de Python.

```python
my_news = [article['title'] for article in data['results']]
```
Es una *list comprehension*. La respuesta de la API tiene una clave `results` que contiene una lista de artículos, cada uno representado como diccionario. Esta línea recorre esa lista y extrae solo el campo `title` de cada artículo. El resultado es una lista de 50 títulos (strings).

## 2. Preprocesamiento de los títulos

```python
STOPWORDS = {"a", "an", "the", "and", "or", "in", "on", "at", "to", "for", "with", "is", "of", "it"}
```
Define un conjunto (`set`) de *stopwords*: palabras muy frecuentes en inglés que aportan poco significado (artículos, preposiciones, conjunciones). Se usa un `set` y no una lista porque verificar si un elemento pertenece a un conjunto es mucho más rápido.

```python
my_news_new = []
```
Crea una lista vacía donde se guardarán los títulos ya limpios.

```python
for text in my_news:
```
Recorre cada título de la lista original. En cada iteración, la variable `text` contiene un título.

```python
    text = text.lower()
```
Convierte todo el texto a minúsculas, para que "Rocket" y "rocket" se traten como la misma palabra.

```python
    text = re.sub(r"https?://\S+", "", text)
```
Elimina URLs. El patrón significa:
- `http` seguido opcionalmente de `s` (`s?`),
- luego `://`,
- luego uno o más caracteres que no sean espacio (`\S+`).

Todo lo que coincide se reemplaza por una cadena vacía. La `r` antes de las comillas indica *raw string*, para que las barras invertidas no se interpreten como caracteres de escape.

```python
    text = re.sub(r"it's", "it is", text)
    text = re.sub(r"i'm", "i am", text)
```
Expanden contracciones comunes del inglés. Dos observaciones:
- Solo funcionan con el apóstrofo recto `'`. Muchos titulares usan el apóstrofo tipográfico `’`, y en ese caso no habrá coincidencia.
- En este código el efecto es casi nulo, porque "it" e "is" están en las stopwords y se eliminan más adelante.

```python
    text = re.sub(r"\W", " ", text)
```
Reemplaza por un espacio todo carácter que no sea "de palabra". `\W` es cualquier cosa distinta de letras, dígitos y guion bajo (`_`). Así se eliminan signos de puntuación, comillas, guiones, etc.

```python
    text = re.sub(r"\d", " ", text)
```
Reemplaza cada dígito (`\d`, del 0 al 9) por un espacio. Esto elimina números como años o nombres de misiones ("Starlink 10" queda como "starlink").

```python
    words = text.split()
```
Tokeniza el texto: `split()` sin argumentos separa por cualquier cantidad de espacios en blanco y descarta los vacíos. El resultado es una lista de palabras.

```python
    filtered_words = [w for w in words if w not in STOPWORDS and len(w) > 1]
```
Otra *list comprehension* que conserva solo las palabras que cumplen dos condiciones:
1. No están en el conjunto de stopwords.
2. Tienen más de una letra (elimina restos como "s" que quedan, por ejemplo, de "nasa's" después de quitar el apóstrofo).

```python
    clean_text = ' '.join(filtered_words)
```
Vuelve a unir las palabras filtradas en un solo string, separadas por un espacio.

```python
    my_news_new.append(clean_text)
```
Agrega el título limpio a la lista de resultados.

## Salida

```python
for i, tw in enumerate(my_tweets_new[:10], 1):
```
Pretende recorrer los primeros 10 elementos (`[:10]`) de la lista limpia. `enumerate(..., 1)` entrega pares (índice, elemento) empezando a contar desde 1 en lugar de 0.

**Aquí hay un error:** la variable `my_tweets_new` no existe; la lista creada se llama `my_news_new`. Al ejecutar, Python lanzará `NameError: name 'my_tweets_new' is not defined`. Probablemente el código se adaptó de un ejercicio anterior con tweets. La línea correcta es:

```python
for i, tw in enumerate(my_news_new[:10], 1):
```

```python
    print(f"{i} : {tw}")
```
Imprime cada título usando un *f-string*, que inserta los valores de `i` y `tw` dentro del texto. La salida tendrá la forma:

```
1 : spacex launches starlink satellites falcon rocket
2 : nasa artemis mission delayed
...
```

## Resumen del flujo

1. Descarga 50 titulares desde una API pública de noticias espaciales.
2. Limpia cada titular: minúsculas, sin URLs, sin puntuación, sin números.
3. Tokeniza y elimina stopwords y palabras de una letra.
4. Muestra los 10 primeros resultados (una vez corregido el nombre de la variable).

Este tipo de preprocesamiento es el paso previo habitual para tareas como conteo de frecuencias, nubes de palabras, vectorización (bag of words, TF-IDF) o clasificación de texto.

https://www.json.org/json-en.html

# Tutorial: qué es JSON y para qué sirve

## 1. Qué es JSON

JSON (JavaScript Object Notation) es un formato de texto para guardar e intercambiar datos estructurados. Es texto plano: cualquier programa, en cualquier lenguaje, puede leerlo y escribirlo.

Nació a partir de la sintaxis de objetos de JavaScript, pero hoy es independiente del lenguaje. Python, Java, R, C# y prácticamente todos los lenguajes modernos lo soportan.

La idea central es simple: representar datos como pares **clave: valor**, igual que una ficha con campos.

```json
{
  "nombre": "Ana",
  "edad": 28,
  "ciudad": "Santiago"
}
```

Si conoces los diccionarios de Python, ya entiendes el 90% de JSON: se ven casi iguales.

## 2. Sintaxis

JSON tiene solo dos estructuras y seis tipos de valor. Con eso se representa cualquier dato.

### Las dos estructuras

**Objeto:** pares clave-valor entre llaves `{ }`. Las claves siempre son strings entre comillas dobles.

```json
{ "titulo": "Lanzamiento Falcon 9", "exitoso": true }
```

**Arreglo:** lista ordenada de valores entre corchetes `[ ]`.

```json
["Marte", "Luna", "Europa"]
```

### Los seis tipos de valor

| Tipo JSON | Ejemplo | Equivalente en Python |
| --- | --- | --- |
| string | `"hola"` | `str` |
| número | `42`, `3.14` | `int`, `float` |
| booleano | `true`, `false` | `True`, `False` |
| nulo | `null` | `None` |
| objeto | `{"a": 1}` | `dict` |
| arreglo | `[1, 2, 3]` | `list` |

### Anidamiento

Los objetos y arreglos pueden contenerse unos a otros, lo que permite datos complejos:

```json
{
  "curso": "Introducción a la Programación",
  "estudiantes": [
    { "nombre": "Ana", "notas": [6.5, 7.0] },
    { "nombre": "Luis", "notas": [5.8, 6.2] }
  ],
  "activo": true,
  "profesor_auxiliar": null
}
```

### Reglas que no se pueden romper

- Las claves y los strings usan **comillas dobles**. Las comillas simples no son válidas.
- No se permite una coma después del último elemento.
- No hay comentarios.
- `true`, `false` y `null` van en minúscula.

## 3. Para qué sirve

JSON es el idioma común con que los programas se pasan datos entre sí. Sus usos principales son tres.

**Comunicación con APIs.** Cuando una aplicación pide datos a un servidor (el clima, noticias, precios, publicaciones), el servidor casi siempre responde en JSON. Una app de Android escrita en Kotlin puede hablar con un servidor en Python porque ambos entienden JSON.

**Archivos de configuración.** Muchas herramientas guardan sus ajustes en JSON. Por ejemplo, `package.json` en proyectos de JavaScript o `settings.json` en VS Code. Incluso los notebooks de Jupyter (`.ipynb`) son archivos JSON por dentro.

**Almacenamiento de datos.** Bases de datos como MongoDB o Firebase guardan documentos en formato JSON (o uno muy parecido). También es común guardar resultados de un programa en un archivo `.json` para usarlos después.

### Por qué se volvió tan popular

- Es legible por personas, no solo por máquinas.
- Es liviano: tiene menos símbolos que alternativas como XML.
- Es universal: todos los lenguajes tienen herramientas para leerlo.
- Su estructura calza directamente con diccionarios y listas, que todo programador ya usa.

## 4. JSON en Python

Python incluye el módulo `json`, que convierte entre texto JSON y estructuras de Python. Hay cuatro funciones clave; la "s" final significa *string*.

| Función | Convierte | Desde / hacia |
| --- | --- | --- |
| `json.loads()` | JSON a Python | un string |
| `json.dumps()` | Python a JSON | un string |
| `json.load()` | JSON a Python | un archivo |
| `json.dump()` | Python a JSON | un archivo |

### De texto JSON a Python

```python
import json

texto = '{"nombre": "Ana", "edad": 28, "activa": true}'
datos = json.loads(texto)

print(type(datos))       # <class 'dict'>
print(datos["nombre"])   # Ana
print(datos["activa"])   # True  (true se convirtió en True)
```

### De Python a texto JSON

```python
alumno = {"nombre": "Luis", "notas": [5.8, 6.2], "tutor": None}

texto = json.dumps(alumno, indent=2, ensure_ascii=False)
print(texto)
```

`indent=2` agrega saltos de línea y sangría para que sea legible. `ensure_ascii=False` mantiene tildes y ñ tal cual, en vez de convertirlas en códigos como `\u00f1`.

### Guardar y leer archivos

```python
# Guardar
with open("alumnos.json", "w", encoding="utf-8") as f:
    json.dump(alumno, f, indent=2, ensure_ascii=False)

# Leer
with open("alumnos.json", "r", encoding="utf-8") as f:
    alumno_leido = json.load(f)
```

### Atajo con requests

Al consultar una API con `requests`, el método `.json()` hace el `json.loads()` por ti:

```python
import requests

respuesta = requests.get("https://api.ejemplo.com/datos")
datos = respuesta.json()   # equivale a json.loads(respuesta.text)
```

## 5. Ejemplo práctico: noticias espaciales

La API de Spaceflight News devuelve un objeto JSON con metadatos y una lista de artículos en la clave `results`. Esta es su forma, simplificada:

```json
{
  "count": 30000,
  "next": "https://api.spaceflightnewsapi.net/v4/articles/?limit=50&offset=50",
  "results": [
    {
      "id": 101,
      "title": "NASA prepares next Artemis mission",
      "news_site": "SpaceNews",
      "published_at": "2026-11-02T14:30:00Z",
      "authors": [{ "name": "Jane Doe" }]
    },
    {
      "id": 102,
      "title": "SpaceX launches new Starlink batch",
      "news_site": "Spaceflight Now",
      "published_at": "2026-11-02T09:10:00Z",
      "authors": [{ "name": "John Smith" }]
    }
  ]
}
```

Para llegar a un dato se sigue el camino desde afuera hacia adentro, usando una clave para cada objeto y un índice para cada arreglo:

```python
import requests

url = "https://api.spaceflightnewsapi.net/v4/articles/?limit=50"
data = requests.get(url).json()

print(data["count"])                              # número total de artículos
print(data["results"][0]["title"])                # título del primer artículo
print(data["results"][0]["authors"][0]["name"])   # primer autor del primer artículo

# Todos los títulos
titulos = [articulo["title"] for articulo in data["results"]]
```

Leída en voz alta, `data["results"][0]["title"]` significa: en `data`, toma la lista `results`; de ella, el elemento 0; de ese artículo, su `title`.

Cuando no conozcas la estructura de una respuesta, explórala así:

```python
print(data.keys())                  # claves del nivel superior
print(data["results"][0].keys())    # claves de un artículo
```

## 6. Errores comunes

| Error | Ejemplo incorrecto | Corrección |
| --- | --- | --- |
| Comillas simples | `{'nombre': 'Ana'}` | `{"nombre": "Ana"}` |
| Coma final | `[1, 2, 3,]` | `[1, 2, 3]` |
| Booleanos al estilo Python | `{"activo": True}` | `{"activo": true}` |
| `None` en vez de `null` | `{"tutor": None}` | `{"tutor": null}` |
| Clave sin comillas | `{nombre: "Ana"}` | `{"nombre": "Ana"}` |
| Confundir string con dict | `texto["nombre"]` sobre un string | `json.loads(texto)["nombre"]` |

Cuando el texto no es JSON válido, Python lanza `json.JSONDecodeError` e indica la línea y columna del problema. Otro error frecuente es `KeyError`: aparece al pedir una clave que no existe. Para evitarlo usa `.get()`, que devuelve `None` en lugar de fallar:

```python
articulo.get("subtitle")          # None si no existe
articulo.get("subtitle", "")      # "" si no existe
```

## 7. Ejercicios propuestos

1. Escribe a mano un JSON que describa tu curso: nombre, año, lista de 3 estudiantes con nombre y una lista de notas cada uno.
2. Encuentra y corrige los 4 errores de este JSON:

   ```
   {'ciudad': "Santiago", poblacion: 7000000, "capital": True, "comunas": ["Ñuñoa", "Maipú",]}
   ```
3. Con `json.loads()`, convierte tu JSON del ejercicio 1 en un diccionario e imprime el promedio de notas de cada estudiante.
4. Usando la API de Spaceflight News, imprime el título y el sitio (`news_site`) de los 10 artículos más recientes.
5. Guarda esos 10 artículos en un archivo `noticias.json` con `json.dump()`, ábrelo en un editor de texto y revisa su estructura.
6. Desafío: cuenta cuántos artículos publicó cada `news_site` entre los 50 descargados y muestra el resultado ordenado de mayor a menor.

# Tutorial: stopwords en español e inglés

## 1. Qué son las stopwords

Las stopwords (palabras vacías) son palabras muy frecuentes en un idioma que aportan poco significado por sí solas: artículos, preposiciones, conjunciones, pronombres y verbos auxiliares.

| Categoría | Inglés | Español |
| --- | --- | --- |
| Artículos | the, a, an | el, la, los, las, un, una |
| Preposiciones | in, on, at, of, to | en, de, a, por, para, con |
| Conjunciones | and, or, but | y, o, pero, que |
| Pronombres | it, he, they, this | él, ella, esto, se, lo |
| Auxiliares | is, are, was, have | es, está, fue, ha |

Si cuentas las palabras de cualquier texto, las más frecuentes casi siempre son stopwords. En inglés, "the" suele ocupar el primer lugar; en español, "de" y "la".

## 2. Para qué se eliminan

Eliminar stopwords deja solo las palabras con contenido, lo que ayuda en tareas como:

- **Conteo de frecuencias y nubes de palabras:** sin filtrar, la nube muestra "the", "de" y "la" en gigante y nada útil.
- **Búsqueda de temas:** identificar de qué trata un conjunto de textos.
- **Clasificación con bolsa de palabras o TF-IDF:** reduce el tamaño del vocabulario y el ruido.

Ejemplo:

```
Original:   "La NASA anunció el lanzamiento de la misión Artemis a la Luna"
Sin stopwords: "nasa anunció lanzamiento misión artemis luna"
```

El significado principal se conserva con menos de la mitad de las palabras.

## 3. Listas listas para usar con NLTK

Escribir la lista a mano (como en el notebook de noticias) sirve para aprender, pero es incompleta. La librería NLTK trae listas para más de 20 idiomas.

```python
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

sw_en = set(stopwords.words('english'))
sw_es = set(stopwords.words('spanish'))

print(len(sw_en), "stopwords en inglés")
print(len(sw_es), "stopwords en español")

print(sorted(sw_es)[:20])   # primeras 20 en orden alfabético
```

Se convierten a `set` porque verificar si una palabra pertenece a un conjunto es mucho más rápido que buscarla en una lista.

Para ver qué idiomas están disponibles:

```python
print(stopwords.fileids())
```

## 4. Función de limpieza

Esta función sirve para ambos idiomas. Recibe un texto y el conjunto de stopwords a usar.

```python
import re

def limpiar(texto, stopwords_set):
    texto = texto.lower()
    texto = re.sub(r"https?://\S+", "", texto)   # quitar URLs
    texto = re.sub(r"\W", " ", texto)            # quitar puntuación
    texto = re.sub(r"\d", " ", texto)            # quitar números
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stopwords_set and len(p) > 1]
    return " ".join(palabras)

print(limpiar("The rocket is ready for the launch to Mars", sw_en))
# rocket ready launch mars

print(limpiar("El cohete está listo para el lanzamiento a Marte", sw_es))
# cohete listo lanzamiento marte
```

En Python 3, `\W` reconoce letras con tilde y la ñ como parte de las palabras, así que "lanzamiento", "está" o "niño" no se cortan.

## 5. Diferencias entre español e inglés

### Tildes

Las listas de NLTK en español traen las palabras con tilde ("está", "él", "más"). Si el texto viene sin tildes, por ejemplo de redes sociales, "esta" sí se elimina pero "mas" no. Una solución es quitar tildes tanto del texto como de la lista:

```python
import unicodedata

def quitar_tildes(texto):
    normalizado = unicodedata.normalize("NFD", texto)
    return "".join(c for c in normalizado if unicodedata.category(c) != "Mn")

sw_es_sin_tildes = {quitar_tildes(p) for p in sw_es}
```

Cuidado: esto también convierte "ñ" en "n", de modo que "año" y "ano" quedan iguales.

### Contracciones

En inglés las contracciones son frecuentes ("it's", "don't", "I'm"). Al quitar la puntuación, "don't" se separa en "don" y "t". Por eso conviene expandirlas antes de limpiar, o filtrar palabras de una letra con `len(p) > 1`.

En español hay solo dos contracciones, "al" y "del", y ambas ya están en la lista de stopwords.

### Variaciones de género y número

El español tiene más formas por palabra ("el, la, los, las", "este, esta, estos, estas"). Por eso su lista de stopwords es más larga que la del inglés.

## 6. Textos con ambos idiomas

Si el corpus mezcla idiomas, se pueden unir las dos listas:

```python
sw_bilingue = sw_en | sw_es   # unión de conjuntos

print(limpiar("Great news: la misión fue un éxito", sw_bilingue))
# great news misión éxito
```

Hay que revisar los choques entre idiomas. Por ejemplo, "a" es stopword en ambos, sin problema. Pero "con" es stopword en español y en inglés puede ser una palabra con sentido ("pros and cons").

## 7. Personalizar la lista

Las listas son un punto de partida. Casi siempre conviene ajustarlas al problema.

**Agregar palabras del dominio.** En noticias espaciales, palabras como "space" o "news" aparecen en casi todos los titulares y no distinguen un tema de otro:

```python
sw_noticias = sw_en | {"space", "news", "new", "says", "update"}
```

**Quitar palabras que sí importan.** En análisis de sentimiento, eliminar "no" o "not" invierte el significado:

```python
print(limpiar("This mission is not a success", sw_en))
# mission success   (el sentido quedó al revés)

sw_sentimiento = sw_en - {"no", "not", "nor"}
print(limpiar("This mission is not a success", sw_sentimiento))
# mission not success
```

## 8. Cuándo no eliminarlas

- **Análisis de sentimiento:** las negaciones cambian el sentido de la frase.
- **Modelos de lenguaje modernos (BERT, GPT y similares):** usan el contexto completo y funcionan mejor con el texto original.
- **Búsqueda de frases exactas:** "To be or not to be" queda vacía si se filtran stopwords.
- **Traducción automática y generación de texto:** se necesita la gramática completa.

Regla práctica: se eliminan cuando el análisis cuenta palabras (frecuencias, nubes, bolsa de palabras, TF-IDF) y se conservan cuando el modelo necesita entender frases.

## 9. Aplicación al notebook de noticias

Reemplazando la lista escrita a mano por la de NLTK:

```python
import requests
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
sw_en = set(stopwords.words('english'))

url = "https://api.spaceflightnewsapi.net/v4/articles/?limit=50"
data = requests.get(url).json()
my_news = [article["title"] for article in data["results"]]

my_news_new = [limpiar(titulo, sw_en) for titulo in my_news]

for i, titulo in enumerate(my_news_new[:10], 1):
    print(f"{i} : {titulo}")
```

## 10. Ejercicios propuestos

1. Imprime las 30 primeras stopwords en español de NLTK. Busca tres que te sorprendan y explica por qué podrían estar en la lista.
2. Limpia la frase "No me gusta que el lanzamiento se haya retrasado" con `sw_es`. Describe qué información se perdió.
3. Crea una versión de `sw_es` sin "no", "ni" y "nunca", y repite el ejercicio anterior.
4. Cuenta las 10 palabras más frecuentes en los 50 titulares de noticias, primero sin filtrar y luego filtrando stopwords. Compara ambas listas. Pista: usa `collections.Counter`.
5. Agrega a la lista dos o tres palabras del dominio espacial que aparezcan en casi todos los titulares y vuelve a contar. ¿Qué temas aparecen ahora?
6. Desafío: escribe cinco frases en español sin tildes (como en un chat) y verifica cuáles stopwords se escapan del filtro. Corrígelo con `quitar_tildes`.

# SIC 2026 · Inteligencia Artificial
## Clase 11: Del scraping a las reseñas: preprocesamiento, TF-IDF y primeras aplicaciones de PLN

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

Ayer terminamos el ejercicio de países y llegamos hasta el scraping, donde la página de Samsung nos ganó. Hoy partimos justo ahí: entendemos qué pasó, conseguimos **reseñas reales** de productos Samsung desde una API y con ellas recorremos el resto del flujo de PLN.

| Hora | Parte | Qué vamos a hacer |
|---|---|---|
| 18:00 a 18:25 | 1. Lo que pasó con Samsung | Páginas estáticas y dinámicas, y cómo saber cuál es cuál |
| 18:25 a 19:00 | 2. Datos desde una API | Descargar reseñas reales en JSON y explorarlas |
| 19:00 a 19:30 | 3. Expresiones regulares | Retomar el ejercicio de porcentajes y limpiar reseñas |
| 19:30 a 19:40 | Pausa | |
| 19:40 a 20:30 | 4. Preprocesamiento | Minúsculas, stopwords, stemming y lematización |
| 20:30 a 21:10 | 5. TF-IDF | Convertir textos en números |
| 21:10 a 21:55 | 6. Aplicaciones | ¿De qué hablan las reseñas? ¿Son positivas o negativas? |
| 21:55 a 22:00 | Cierre | |

Cada paso tiene la misma estructura: **qué vamos a hacer**, **el código** y **qué deberías ver**. Los ejercicios tienen verificación y las soluciones están al final.

**Antes de empezar:** *Archivo > Guardar una copia en Drive*.

In [ ]:
import os
import nltk
nltk.download('stopwords')


# Buscar la ruta del recurso 'stopwords'
ruta_stopwords = nltk.data.find('corpora/stopwords')
print("Los archivos están en:", ruta_stopwords)

# Listar los archivos de idiomas disponibles dentro de esa carpeta
print("Idiomas disponibles:", os.listdir(ruta_stopwords))

Los archivos están en: /root/nltk_data/corpora/stopwords
Idiomas disponibles: ['hinglish', 'belarusian', 'basque', 'dutch', 'indonesian', 'azerbaijani', 'romanian', 'chinese', 'README', 'slovene', 'swedish', 'arabic', 'finnish', 'danish', 'hebrew', 'albanian', 'german', 'kazakh', 'tamil', 'spanish', 'greek', 'catalan', 'english', 'hungarian', 'italian', 'turkish', 'french', 'bengali', 'nepali', 'uzbek', 'norwegian', 'portuguese', 'tajik', 'russian']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
import nltk
from nltk.corpus import stopwords

# Aseguramos la descarga de las stopwords
nltk.download('stopwords', quiet=True)

# Obtenemos los conjuntos de palabras
stopwords_es = sorted(list(stopwords.words('spanish')))
stopwords_en = sorted(list(stopwords.words('english')))

print(f"--- STOPWORDS EN ESPAÑOL ({len(stopwords_es)} palabras) ---")
print(stopwords_es[:50], "... y más.")  # Mostramos las primeras 50

print(f"\n--- STOPWORDS EN INGLÉS ({len(stopwords_en)} palabras) ---")
print(stopwords_en[:50], "... y más.")

--- STOPWORDS EN ESPAÑOL (313 palabras) ---
['a', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'como', 'con', 'contra', 'cual', 'cuando', 'de', 'del', 'desde', 'donde', 'durante', 'e', 'el', 'ella', 'ellas', 'ellos', 'en', 'entre', 'era', 'erais', 'eran', 'eras', 'eres', 'es', 'esa', 'esas', 'ese', 'eso', 'esos', 'esta', 'estaba', 'estabais', 'estaban', 'estabas', 'estad', 'estada', 'estadas', 'estado', 'estados', 'estamos', 'estando', 'estar', 'estaremos', 'estará'] ... y más.

--- STOPWORDS EN INGLÉS (198 palabras) ---
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn'] ... y más.


In [ ]:
import spacy

# Descargamos el modelo en español si no está instalado
try:
    nlp = spacy.load("es_core_news_sm")
except OSError:
    print("Descargando el modelo en español de spaCy...")
    spacy.cli.download("es_core_news_sm")
    nlp = spacy.load("es_core_news_sm")

texto_ejemplo = "Los gatos corrieron rápidamente por el jardín y fueron muy felices."
doc = nlp(texto_ejemplo)

print(f"{'Palabra':15} | {'Lema (Diccionario)':18} | {'Categoría Gramatical (POS)':25}")
print("-" * 65)
for token in doc:
    # Filtramos signos de puntuación
    if not token.is_punct:
        print(f"{token.text:15} | {token.lemma_:18} | {token.pos_:25}")

Descargando el modelo en español de spaCy...
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Palabra         | Lema (Diccionario) | Categoría Gramatical (POS)
-----------------------------------------------------------------
Los             | el                 | DET                      
gatos           | gato               | NOUN                     
corrieron       | correr             | VERB                     
rápidamente     | rápidamente        | ADV                      
por             | por                | ADP                      
el              | el                 | DET                      
jardín          | jardín             | NOUN                     
y               | y            

In [ ]:
import spacy

# Cargamos el modelo en español para verificar sus stopwords
nlp = spacy.load("es_core_news_sm")
stopwords_spacy = nlp.Defaults.stop_words

print(f"Cantidad de stopwords en español en spaCy: {len(stopwords_spacy)}")
# Mostramos algunas de ellas como ejemplo
print("Ejemplos:", sorted(list(stopwords_spacy))[:20])

Cantidad de stopwords en español en spaCy: 521
Ejemplos: ['a', 'acuerdo', 'adelante', 'ademas', 'además', 'afirmó', 'agregó', 'ahi', 'ahora', 'ahí', 'al', 'algo', 'alguna', 'algunas', 'alguno', 'algunos', 'algún', 'alli', 'allí', 'alrededor']


# Explicación: tokenización, stemming y lematización con NLTK

Este notebook muestra tres técnicas básicas de procesamiento de lenguaje natural usando la librería NLTK (Natural Language Toolkit) sobre un fragmento del discurso de Barack Obama del 4 de noviembre de 2008.

## 0. Preparación

```python
import nltk
```
Importa NLTK, una de las librerías clásicas de Python para trabajar con texto.

```python
# nltk.download('punkt')
# nltk.download('wordnet')
# nltk.download('punkt_tab')
```
Líneas comentadas (el `#` hace que Python las ignore). Quedan como recordatorio de qué recursos hay que descargar.

```python
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')
```
Las mismas líneas, ahora activas. Descargan los recursos que NLTK necesita y que no vienen con la instalación:

- `punkt`: modelo para separar texto en oraciones y palabras.
- `punkt_tab`: versión nueva de Punkt, exigida desde NLTK 3.9. Tener ambas asegura que funcione en cualquier versión.
- `wordnet`: base de datos léxica del inglés, necesaria para lematizar.

En Colab hay que volver a ejecutar esta celda cada vez que el entorno se reinicia.

## 1. El texto de entrada

```python
paragraph = """If there is anyone out there who still doubts..."""
```
Guarda el fragmento del discurso en la variable `paragraph`. Las triples comillas `"""` permiten escribir un string de varias líneas. Es un buen texto de prueba porque tiene oraciones largas, contracciones ("It's", "he's", "they've"), puntuación variada ("--", ";", ":") y nombres propios.

## 2. Tokenización

Tokenizar es dividir un texto en unidades más pequeñas (tokens). Es el primer paso de casi cualquier análisis de texto.

```python
sentences = nltk.sent_tokenize(paragraph)
print(sentences)
```
Divide el párrafo en una **lista de oraciones**. El modelo Punkt sabe distinguir un punto que termina una oración de uno que no lo hace, como en "Dr." o "U.S.". El resultado es una lista de strings, uno por oración.

```python
words = nltk.word_tokenize(paragraph)
print(words)
```
Divide el párrafo en una **lista de palabras y signos**. A diferencia de `split()`, que solo corta por espacios:

| Texto | Con `split()` | Con `word_tokenize()` |
| --- | --- | --- |
| `answer.` | `answer.` | `answer`, `.` |
| `It's` | `It's` | `It`, `'s` |
| `America!` | `America!` | `America`, `!` |

Los signos de puntuación quedan como tokens separados y las contracciones se dividen en sus partes.

## 3. Stemming

El stemming recorta las palabras hasta su **raíz** (stem) aplicando reglas fijas para quitar sufijos. Es rápido, pero mecánico: no consulta un diccionario, así que la raíz resultante muchas veces no es una palabra real.

```python
from nltk.stem import PorterStemmer, LancasterStemmer, SnowballStemmer
```
Importa tres algoritmos de stemming:

- **Porter:** el más clásico (1980) y el más usado. Moderado.
- **Lancaster:** más agresivo; recorta más y a veces deja raíces muy cortas.
- **Snowball:** versión mejorada de Porter. Tiene la ventaja de funcionar en varios idiomas, incluido el español: `SnowballStemmer('spanish')`.

```python
#stemmer = SnowballStemmer('english')
#stemmer = LancasterStemmer()
stemmer = PorterStemmer()
```
Se elige uno. Los otros dos quedan comentados para poder probarlos cambiando el `#` de lugar.

```python
for i in range(len(sentences)):
    words = nltk.word_tokenize(sentences[i])
    words = [stemmer.stem(x) for x in words]
    sentences[i] = ' '.join(words)
```
Recorre cada oración por su índice `i` y hace tres cosas:

1. Tokeniza la oración en palabras.
2. Aplica `stemmer.stem()` a cada palabra con una list comprehension.
3. Une las palabras con espacios y **reemplaza** la oración original en la lista.

Importante: este ciclo sobrescribe `sentences`. Después de ejecutarlo, las oraciones originales se pierden; por eso más adelante se vuelven a generar.

```python
print(sentences)
```
Muestra las oraciones procesadas. Se verán palabras como "doubt", "possibl", "democraci" o "elect".

```python
stemmer.stem("worked")        # 'work'
stemmer.stem("cried")         # 'cri'
stemmer.stem("remembrance")   # 'remembr'
stemmer.stem("happily")       # 'happili'
```
Ejemplos sueltos con Porter. "worked" queda bien, pero "cri", "remembr" y "happili" no son palabras reales. Para contar frecuencias esto no importa: lo relevante es que "cried" y "cries" terminen en la misma raíz. Además, Porter convierte todo a minúsculas.

En Colab solo se muestra el resultado de la **última línea** de una celda. Si estas cuatro líneas están en la misma celda, solo aparecerá `'happili'`; para ver todas, conviene envolverlas en `print()`.

## 4. Lematización

La lematización lleva cada palabra a su **lema**: la forma que aparecería en el diccionario. A diferencia del stemming, consulta un diccionario (WordNet), así que el resultado siempre es una palabra real.

```python
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
lemmatizer = WordNetLemmatizer()
```
Importa el lematizador basado en WordNet y el módulo `wordnet`, que se usará para indicar categorías gramaticales. Luego crea el objeto `lemmatizer`. WordNet solo funciona para inglés.

```python
sentences = nltk.sent_tokenize(paragraph)
```
Vuelve a generar las oraciones originales, porque el ciclo de stemming las había sobrescrito.

```python
for i in range(len(sentences)):
    words = nltk.word_tokenize(sentences[i])
    words = [lemmatizer.lemmatize(x) for x in words]
    sentences[i] = ' '.join(words)

print(sentences)
```
El mismo ciclo que antes, pero con `lemmatize()` en lugar de `stem()`. Se notará que cambian menos palabras: "States" pasa a "State", "hours" a "hour", pero verbos como "waited" quedan igual.

Esto ocurre porque, si no se le indica la categoría gramatical, `lemmatize()` **asume que toda palabra es un sustantivo**. Eso produce errores curiosos: "has" se convierte en "ha" y "was" en "wa", porque el lematizador los trata como plurales de sustantivos.

## 5. Lematización con categoría gramatical

```
Some more examples with the POS tags specified.
```
Esta línea es texto explicativo, no código ("Algunos ejemplos más con la categoría gramatical especificada"). En el notebook original debe ir en una celda de texto; si queda en una celda de código, Python lanzará un `SyntaxError`.

POS significa *Part of Speech*, la categoría gramatical de la palabra. Se indica como segundo argumento:

```python
lemmatizer.lemmatize("dogs", wordnet.NOUN)      # 'dog'
lemmatizer.lemmatize("wolves", wordnet.NOUN)    # 'wolf'
lemmatizer.lemmatize("has", wordnet.VERB)       # 'have'
lemmatizer.lemmatize("helpful", wordnet.ADJ)    # 'helpful'
lemmatizer.lemmatize("happily", wordnet.ADV)    # 'happily'
```

- **"dogs" a "dog":** plural regular.
- **"wolves" a "wolf":** plural irregular. Un stemmer no lo resolvería bien, porque no hay una regla simple que convierta "ves" en

Las tres líneas crean un "stemmer", un objeto que recorta palabras hasta su raíz. Solo una está activa (la que no tiene `#`), y en este caso es **Lancaster**. Todo el código posterior que use `stemmer.stem(...)` aplicará ese algoritmo. Para probar otro, se mueve el `#` de lugar.

La diferencia entre los tres está en qué tan agresivamente recortan.

## Porter

Creado por Martin Porter en 1980. Es el algoritmo clásico y el más citado. Aplica unas 60 reglas en cinco pasos ordenados, quitando sufijos como "-ing", "-ed", "-ness" o "-ational". Es moderado, pero deja raíces raras con algunas terminaciones, como "-ly":

```python
PorterStemmer().stem("generously")   # 'gener'
PorterStemmer().stem("fairly")       # 'fairli'
```

Solo funciona en inglés.

## Snowball

Es la versión mejorada de Porter, creada por el mismo autor alrededor de 2001 (por eso también se le llama "Porter2"). Corrige varios defectos del original y da raíces más razonables:

```python
SnowballStemmer('english').stem("generously")   # 'generous'
SnowballStemmer('english').stem("fairly")       # 'fair'
```

Su gran ventaja es que **funciona en varios idiomas**, incluido el español. Por eso exige indicar el idioma entre paréntesis:

```python
SnowballStemmer('spanish').stem("corriendo")
print(SnowballStemmer.languages)   # idiomas disponibles
```

## Lancaster

Desarrollado en la Universidad de Lancaster en 1990. Es el más **agresivo**: aplica sus reglas una y otra vez hasta que ninguna se puede aplicar, así que recorta mucho más. Esto agrupa más palabras bajo una misma raíz, pero a menudo deja raíces cortísimas o que mezclan palabras sin relación:

```python
LancasterStemmer().stem("maximum")     # 'maxim'
LancasterStemmer().stem("provision")   # 'provid'
LancasterStemmer().stem("owed")        # 'ow'
LancasterStemmer().stem("cement")      # 'cem'
```

Solo funciona en inglés.

## Comparación

| Aspecto | Porter | Snowball | Lancaster |
| --- | --- | --- | --- |
| Año | 1980 | ~2001 | 1990 |
| Agresividad | Media | Media, más pulida | Alta |
| Raíces legibles | A veces | Más a menudo | Rara vez |
| Idiomas | Inglés | Varios, incluido español | Inglés |
| Riesgo principal | Raíces extrañas | El menor de los tres | Unir palabras distintas |

Los dos errores típicos del stemming ayudan a entender el contraste:

- **Sobre-stemming:** recortar tanto que palabras con significados distintos quedan con la misma raíz. Es el problema típico de Lancaster.
- **Sub-stemming:** recortar tan poco que variantes de la misma palabra quedan separadas. Es más común en algoritmos conservadores.

## Para verlo en la práctica

Este bloque compara los tres sobre las mismas palabras:

```python
from nltk.stem import PorterStemmer, LancasterStemmer, SnowballStemmer

porter = PorterStemmer()
lancaster = LancasterStemmer()
snowball = SnowballStemmer('english')

palabras = ["worked", "cried", "remembrance", "happily", "generously",
            "fairly", "maximum", "provision", "running", "organization"]

print(f"{'Palabra':<15}{'Porter':<15}{'Snowball':<15}{'Lancaster':<15}")
for p in palabras:
    print(f"{p:<15}{porter.stem(p):<15}{snowball.stem(p):<15}{lancaster.stem(p):<15}")
```

El formato `:<15` alinea cada columna a 15 caracteres para que la salida se lea como tabla.

## Cuál usar

Como regla general, **Snowball** es la mejor opción por defecto: es más preciso que Porter y es el único de los tres que sirve para textos en español. Porter se sigue usando por tradición y para comparar con trabajos anteriores. Lancaster conviene solo cuando se quiere reducir el vocabulario al máximo y se acepta perder precisión.

En lematización no hay tres algoritmos equivalentes como en stemming. La diferencia práctica está en **qué herramienta se usa** y, sobre todo, en **cómo se obtiene la categoría gramatical** de cada palabra, que es lo que determina la calidad del resultado.

## La idea base

El lematizador busca cada palabra en un diccionario y devuelve su forma base (lema). Pero una misma palabra puede tener lemas distintos según su función:

```python
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()

lemmatizer.lemmatize("better")                  # 'better'  (asume sustantivo)
lemmatizer.lemmatize("better", wordnet.ADJ)     # 'good'
lemmatizer.lemmatize("meeting")                 # 'meeting' (sustantivo: "la reunión")
lemmatizer.lemmatize("meeting", wordnet.VERB)   # 'meet'    (verbo: "reuniéndose")
```

Por eso existen tres formas de trabajar, de menor a mayor calidad.

## Opción 1: WordNet sin categoría

Es lo que hace el notebook en el ciclo principal: `lemmatizer.lemmatize(x)`. Trata todo como sustantivo, así que solo resuelve plurales ("dogs" a "dog") y deja los verbos sin cambiar o los deforma ("has" a "ha", "was" a "wa"). Es rápido pero pobre.

## Opción 2: WordNet con categoría automática

Se combina con un etiquetador gramatical (POS tagger) que detecta la función de cada palabra en la oración. NLTK trae uno, pero usa etiquetas distintas a las de WordNet, así que hay que traducirlas:

```python
import nltk
nltk.download('averaged_perceptron_tagger_eng')

from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def a_wordnet(etiqueta):
    if etiqueta.startswith('J'):
        return wordnet.ADJ
    if etiqueta.startswith('V'):
        return wordnet.VERB
    if etiqueta.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

oracion = "He has fought even longer and harder for the country"
tokens = nltk.word_tokenize(oracion)
etiquetas = nltk.pos_tag(tokens)
print(etiquetas)   # [('He', 'PRP'), ('has', 'VBZ'), ('fought', 'VBN'), ...]

lemas = [lemmatizer.lemmatize(palabra, a_wordnet(tag)) for palabra, tag in etiquetas]
print(lemas)
```

Paso a paso:

1. `nltk.pos_tag()` devuelve pares (palabra, etiqueta) con el sistema de etiquetas Penn Treebank: `VBZ` es verbo en tercera persona, `NN` sustantivo, `JJ` adjetivo, `RB` adverbio.
2. La función `a_wordnet()` mira solo la primera letra de la etiqueta y la convierte a la categoría que entiende WordNet.
3. Se lematiza cada palabra con su categoría correcta. Ahora "has" pasa a "have" y "fought" a "fight".

Esto mejora mucho el resultado, pero sigue funcionando solo en inglés.

## Opción 3: spaCy

spaCy es una librería más moderna que hace todo el proceso de una vez: tokeniza, detecta la categoría gramatical y lematiza. Además tiene modelos para muchos idiomas, **incluido el español**.

En inglés (el modelo pequeño ya viene instalado en Colab):

```python
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("He has fought even longer and harder for the country")

for token in doc:
    print(f"{token.text:<10}{token.pos_:<8}{token.lemma_}")
```

En español hay que descargar el modelo primero:

```python
!python -m spacy download es_core_news_sm
```

```python
nlp_es = spacy.load("es_core_news_sm")
doc = nlp_es("Los corredores corrieron rápidamente hacia las montañas")

for token in doc:
    print(f"{token.text:<14}{token.pos_:<8}{token.lemma_}")
```

Si Colab no encuentra el modelo justo después de descargarlo, basta con reiniciar el entorno (Entorno de ejecución > Reiniciar sesión) y volver a cargarlo.

El modelo pequeño (`sm`) comete errores ocasionales, sobre todo con palabras poco comunes. Los modelos más grandes (`es_core_news_md`, `es_core_news_lg`) son más precisos pero pesan más.

## Comparación

| Aspecto | WordNet sin POS | WordNet + pos_tag | spaCy |
| --- | --- | --- | --- |
| Categoría gramatical | Asume sustantivo | Detectada por NLTK | Detectada automáticamente |
| Verbos | Mal ("has" a "ha") | Bien ("has" a "have") | Bien |
| Idiomas | Inglés | Inglés | Muchos, incluido español |
| Código necesario | Una línea | Función de traducción | Pocas líneas |
| Velocidad | Muy rápida | Media | Media |

## Y frente al stemming

| Aspecto | Stemming (Porter, Snowball, Lancaster) | Lematización |
| --- | --- | --- |
| Cómo funciona | Corta sufijos con reglas | Busca en un diccionario |
| Resultado | Raíz, a veces no es palabra ("happili") | Palabra real ("happily") |
| Irregulares | Falla ("wolves", "was", "better") | Acierta ("wolf", "be", "good") |
| Contexto | No lo usa | Lo usa para decidir la categoría |

## Cuál usar

Para inglés en un ejercicio con NLTK, la opción 2 (WordNet con `pos_tag`) es la que muestra bien la idea sin agregar librerías. Para cualquier trabajo en español, o cuando se quiere un resultado confiable con poco código, **spaCy** es la elección estándar. La opción 1 solo sirve para ilustrar por qué la categoría gramatical importa.

---
# Paso 0. Preparar el entorno

**Qué vamos a hacer:** instalar el modelo de español de spaCy, cargar librerías y definir `revisar`. Tarda alrededor de un minuto.

In [ ]:
!python -m spacy download es_core_news_sm -q

import re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download("stopwords", quiet=True)

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    """Imprime si un ejercicio está bien resuelto."""
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

NAVEGADOR = {"User-Agent": "Mozilla/5.0 (curso SIC 2026)", "Accept-Language": "es-CL,es;q=0.9"}
print("Listo.")

---
# Parte 1. Lo que pasó con Samsung

Ayer pedimos con `requests` la página de búsqueda de Samsung. La respuesta fue `200` (todo bien) y trajo más de 300.000 caracteres de HTML, pero `soup.find(...)` devolvió `None`: **los productos no estaban en el HTML**. ¿Por qué?

## Paso 1.1. Páginas estáticas y dinámicas

| | Página estática | Página dinámica |
|---|---|---|
| Cómo se arma | El servidor envía el HTML completo | El servidor envía un "cascarón" y el navegador lo completa ejecutando JavaScript |
| Qué recibe `requests` | Todo el contenido | Solo el cascarón: `requests` no ejecuta JavaScript |
| Ejemplos | Wikipedia, blogs, muchos sitios de gobierno | Tiendas en línea, buscadores, redes sociales |

**Cómo saberlo sin adivinar:** busca en `res.text` algo que ves en la página del navegador (un nombre de producto, o la clase que encontraste con el inspector). Si no está, la página es dinámica.

**Qué vamos a hacer:** aplicar esa prueba a una página estática y a la de Samsung.

In [ ]:
def es_estatica(url, algo_que_deberia_aparecer):
    """Descarga una página y revisa si un texto o una clase aparece en el HTML que recibe requests."""
    try:
        res = requests.get(url, headers=NAVEGADOR, timeout=15)
        presente = algo_que_deberia_aparecer in res.text
        print(f"Estado {res.status_code} | {len(res.text):,} caracteres | ¿aparece '{algo_que_deberia_aparecer}'? {presente}")
        return presente
    except Exception as e:
        print("No se pudo descargar:", type(e).__name__)
        return None

print("Wikipedia:")
es_estatica("https://es.wikipedia.org/wiki/Procesamiento_de_lenguajes_naturales", "lenguaje natural")
print("\nBuscador de Samsung:")
es_estatica("https://www.samsung.com/cl/aisearch/?searchvalue=Lavadora%20de%2017%20kg", "pd21-product-card__name")

**Qué deberías ver:** en Wikipedia, `True`: el texto viene en el HTML. En Samsung, `False`: la clase de las tarjetas de producto no viene, porque se agrega después con JavaScript. (Si la red de Colab falla, verás un mensaje de error: la idea es la misma.)

## Paso 1.2. El otro error: `find` y `find_all`

Ayer había un segundo problema que habría aparecido igual:

- `find` devuelve **el primer** elemento que coincide (o `None` si no hay ninguno).
- `find_all` devuelve **una lista** con todos los que coinciden (vacía si no hay ninguno).

**Qué vamos a hacer:** probar ambos con un HTML pequeño escrito a mano, que sí trae los productos.

In [ ]:
html_ejemplo = """
<div class="tarjeta"><p class="nombre">Lavadora 17 kg</p><span class="precio">$399.990</span></div>
<div class="tarjeta"><p class="nombre">Lavadora 21 kg</p><span class="precio">$549.990</span></div>
<div class="tarjeta"><p class="nombre">Secadora 9 kg</p><span class="precio">$459.990</span></div>
"""
sopa = BeautifulSoup(html_ejemplo, "html.parser")

print("find:    ", sopa.find(class_="nombre"))
print("find_all:", sopa.find_all(class_="nombre"))
print("\nNombres limpios:", [n.get_text(strip=True) for n in sopa.find_all(class_="nombre")])

**Qué deberías ver:** `find` devuelve una sola etiqueta; `find_all`, una lista de tres. Con `get_text(strip=True)` se obtiene solo el texto.

### Ejercicio 1
Con `sopa` y `find_all`, crea `precios`: una **lista de números enteros** con los tres precios (por ejemplo `399990`). Pista: saca el texto y quita `$` y los puntos con `.replace()`.

In [ ]:
precios = None  # TU CÓDIGO AQUÍ
print(precios)

In [ ]:
revisar(lambda: precios == [399990, 549990, 459990])

## Paso 1.3. Tres caminos para una página dinámica

1. **Buscar la API que usa la página** (el mejor). La página dinámica pide sus datos a algún servidor, casi siempre en formato **JSON**. Esa petición se ve en el navegador: F12, pestaña *Network*, filtro *Fetch/XHR*. Si encontramos esa URL, la consultamos directo con `requests`: es rápido y liviano.
2. **Usar un navegador automatizado** (Selenium o Playwright). Abre un navegador real, espera que se ejecute el JavaScript y entrega el HTML completo. Funciona casi siempre, pero es lento y pesado.
3. **Leer datos estructurados de la página** (JSON-LD). Algunas páginas incluyen sus datos principales en un bloque de JSON dentro del HTML. Sirve para nombres y códigos, no siempre para precios.

El año pasado, una estudiante del curso siguió el **camino 1** y encontró la API que Samsung usa para mostrar las reseñas de sus productos. Eso es lo que haremos ahora.

**Buenas prácticas:** son APIs internas del sitio, así que las usamos con fines educativos, con pocas peticiones y una pausa entre ellas.

---
# Parte 2. Reseñas reales desde una API

## Paso 2.1. ¿Qué es una API y qué es JSON?

> **Definición. API:** una dirección a la que un programa le pide datos y recibe una respuesta ordenada, pensada para máquinas y no para personas.
>
> **Definición. JSON:** un formato de texto para datos, con llaves `{}` para objetos (como diccionarios de Python) y corchetes `[]` para listas. `res.json()` lo convierte directo en diccionarios y listas de Python.

Las reseñas de los productos de Samsung Chile las entrega **Bazaarvoice**, un servicio de reseñas. La API recibe el **SKU** del producto (su código) y devuelve hasta 100 reseñas por página.

## Paso 2.2. Descargar las reseñas

**Qué vamos a hacer:** pedir las reseñas de tres productos (relojes y pulseras Galaxy), juntar las páginas, quitar reseñas repetidas y quedarnos con la nota, el título y el texto.

**Si la API no responde** (Samsung puede cambiarla en cualquier momento), el notebook usa un respaldo de 60 reseñas **ficticias** escritas para esta clase, con el mismo formato. Todo lo que sigue funciona con cualquiera de las dos.

In [ ]:
URL_RESENAS = "https://apps.bazaarvoice.com/bfd/v1/clients/samsung-es_cl/api-products/cv2/resources/data/reviews.json"
HEADERS_API = {
    "Accept": "application/json",
    "X-Bazaarvoice-Api-Version": "5.5",
    "X-Bazaarvoice-Platform-Version": "2",
    "Bv-Bfd-Token": "17499,main_site,es_CL",
    "Origin": "https://www.samsung.com",
}
SKUS = ["SM-L705FZB1CHO", "SM-L320NZSALTA", "SM-R390NZAALTA"]

def pagina_de_resenas(sku, offset):
    """Pide una página de hasta 100 reseñas de un producto."""
    parametros = {
        "resource": "reviews", "action": "REVIEWS_N_STATS",
        "filter": [f"productid:eq:{sku}", "contentlocale:eq:es_CL,es*"],
        "apiversion": "5.5", "displaycode": "17499-es_cl",
        "sort": "submissiontime:desc", "limit": 100, "offset": offset,
    }
    res = requests.get(URL_RESENAS, params=parametros, headers=HEADERS_API, timeout=15)
    res.raise_for_status()
    return res.json().get("response", {}).get("Results", [])

def descargar_resenas(skus, max_paginas=2):
    todas = []
    for sku in skus:
        for pagina in range(max_paginas):
            resultados = pagina_de_resenas(sku, offset=pagina * 100)
            print(f"{sku} | página {pagina + 1}: {len(resultados)} reseñas")
            todas.extend(resultados)
            time.sleep(1)                       # pausa para no saturar la API
            if len(resultados) < 100:           # no hay más páginas
                break
    return pd.DataFrame(todas)

In [ ]:
RESPALDO = [
    (5, 'Excelente compañero para correr', 'Lo uso todos los días para correr y el GPS es muy preciso. La batería dura casi dos días con el monitoreo activado.'),
    (5, 'Me encanta el seguimiento del sueño', 'El análisis del sueño es muy completo y me ayudó a ordenar mis horarios. La pantalla se ve perfecta incluso al sol.'),
    (4, 'Muy buen reloj, batería justa', 'Cumple con todo lo que promete, las notificaciones llegan rápido. La batería dura un día y medio, esperaba un poco más.'),
    (5, 'Pantalla espectacular', 'La pantalla es brillante y nítida, se lee perfecto bajo el sol. Muy cómodo para usar todo el día.'),
    (1, 'La batería no dura nada', 'Con el GPS activado la batería no alcanza a durar un día completo. Muy decepcionado para el precio que tiene.'),
    (2, 'Problemas de conexión con el celular', 'Se desconecta del celular varias veces al día y pierdo las notificaciones. Tuve que reinstalar la aplicación dos veces.'),
    (5, 'Ideal para el gimnasio', 'Registra bien los entrenamientos y el ritmo cardíaco. La aplicación muestra todo muy claro.'),
    (4, 'Buen producto, despacho rápido', 'Llegó en dos días y bien embalado. El reloj funciona muy bien, aunque la correa es un poco dura al principio.'),
    (1, 'Dejó de cargar al mes', 'Al mes de uso dejó de cargar. El servicio técnico se demoró tres semanas en responder. No lo recomiendo.'),
    (5, 'El mejor reloj que he tenido', 'Diseño elegante, pantalla grande y la batería dura bastante. Las funciones de salud son muy útiles.'),
    (3, 'Cumple, pero es caro', 'Funciona bien, pero por el precio esperaba más autonomía de batería. La pantalla es muy buena.'),
    (2, 'La aplicación es lenta', 'La aplicación del celular tarda mucho en sincronizar los datos del sueño. A veces no sincroniza nada.'),
    (5, 'Perfecto para monitorear la salud', 'Mide el ritmo cardíaco, el oxígeno y el estrés. Me da tranquilidad poder revisar mi salud cada día.'),
    (4, 'Muy cómodo', 'Casi no se siente en la muñeca y se puede dormir con él. La batería podría durar más.'),
    (1, 'Pantalla rayada en una semana', 'La pantalla se rayó a la semana de uso normal. Esperaba un vidrio más resistente.'),
    (5, 'Excelente calidad precio', 'Lo compré en oferta y es una maravilla. Batería, pantalla y aplicación funcionan muy bien.'),
    (2, 'El GPS falla', 'El GPS pierde la señal en la ciudad y los recorridos quedan cortados. Para correr no me sirve.'),
    (5, 'Notificaciones perfectas', 'Recibo los mensajes y llamadas sin sacar el celular. La conexión con el teléfono es estable.'),
    (4, 'Buen seguimiento del sueño', 'El seguimiento del sueño es bastante preciso. Me gustaría que la aplicación diera más consejos.'),
    (1, 'No reconoce el celular', 'Después de la actualización ya no se conecta con mi celular. El soporte no me dio solución.'),
    (5, 'Muy buena batería', 'Con uso normal la batería me dura casi dos días completos. Carga rápido.'),
    (3, 'Correa incómoda', 'El reloj es bueno pero la correa que trae es incómoda y me irrita la piel. Tuve que comprar otra.'),
    (5, 'Lo recomiendo', 'Funciona perfecto, la pantalla es hermosa y el seguimiento del ejercicio es muy completo.'),
    (2, 'Mide mal el ritmo cardíaco', 'El ritmo cardíaco marca valores muy distintos a los de mi otro dispositivo. No confío en la medición.'),
    (4, 'Bonito y funcional', 'Se ve muy bien y cumple. La batería es lo único que podría mejorar.'),
    (5, 'Gran regalo', 'Se lo regalé a mi papá y lo usa todos los días para caminar. La pantalla es grande y fácil de leer.'),
    (1, 'Muy caro para lo que ofrece', 'Por ese precio esperaba una batería mucho mejor. Tengo que cargarlo todas las noches.'),
    (5, 'Excelente para nadar', 'Lo uso en la piscina sin problemas y registra bien las vueltas. Resistente al agua de verdad.'),
    (4, 'Buena aplicación', 'La aplicación es completa y fácil de usar. A veces la sincronización tarda un poco.'),
    (2, 'Se reinicia solo', 'Se reinicia solo varias veces a la semana. Lo envié al servicio técnico y sigue igual.'),
    (5, 'Pantalla siempre encendida', 'La pantalla siempre encendida es muy útil y la batería igual aguanta todo el día.'),
    (3, 'Bueno, sin más', 'Hace lo básico bien. Nada que me sorprenda, pero tampoco me ha fallado.'),
    (5, 'Mide muy bien el sueño', 'Me sorprendió lo bien que mide las fases del sueño. Ahora duermo mejor.'),
    (1, 'Despacho tardío', 'El despacho se demoró tres semanas y nadie respondía los correos. El producto está bien, la atención pésima.'),
    (4, 'Liviano y rápido', 'El reloj es liviano y la interfaz es muy rápida. La batería es justa si se usa el GPS.'),
    (5, 'Funciones de salud increíbles', 'El electrocardiograma y la presión son funciones muy útiles. Excelente para cuidar la salud.'),
    (2, 'Batería se agota rápido', 'Desde la última actualización la batería se agota en medio día. Antes duraba mucho más.'),
    (5, 'Muy satisfecha', 'Pantalla nítida, buena batería y se conecta sin problemas con mi celular.'),
    (4, 'Recomendable para correr', 'El GPS es preciso y el ritmo cardíaco es confiable. La correa podría ser más cómoda.'),
    (1, 'Llegó defectuoso', 'Llegó con la pantalla con un pixel muerto. El cambio tomó más de un mes.'),
    (5, 'Super cómodo para dormir', 'No molesta para dormir y el análisis del sueño es muy bueno. La batería aguanta bien.'),
    (3, 'Pantalla buena, batería regular', 'La pantalla es excelente pero la batería no me dura el día completo con todo activado.'),
    (5, 'Me ayudó a moverme más', 'Los recordatorios para moverme y el seguimiento de pasos me motivaron a hacer más ejercicio.'),
    (2, 'La conexión bluetooth falla', 'Se pierde la conexión bluetooth con el celular constantemente. Muy frustrante.'),
    (4, 'Muy buen diseño', 'El diseño es elegante y la pantalla se ve muy bien. La aplicación podría ser más intuitiva.'),
    (5, 'Excelente para entrenar', 'Registra todo tipo de ejercicio y el ritmo cardíaco es exacto. Muy contento con la compra.'),
    (1, 'Servicio técnico pésimo', 'El reloj falló y el servicio técnico no me dio solución en dos meses. No vuelvo a comprar.'),
    (5, 'Batería de dos días', 'La batería dura dos días completos con uso normal. Muy contento con la autonomía.'),
    (4, 'Sincroniza bien', 'Se sincroniza bien con la aplicación del celular y las notificaciones llegan al instante.'),
    (2, 'Correa se rompió', 'La correa se rompió al segundo mes. Para un producto de este precio es inaceptable.'),
    (5, 'Pantalla y salud excelentes', 'La pantalla es preciosa y las funciones de salud son muy completas. Lo uso para controlar mi presión.'),
    (3, 'Precio alto', 'Es un buen reloj, pero el precio es demasiado alto comparado con otros modelos.'),
    (5, 'Perfecto', 'Todo funciona perfecto: batería, pantalla, sueño y ejercicio. Muy recomendable.'),
    (1, 'No sincroniza el sueño', 'Los datos del sueño nunca aparecen en la aplicación. Probé de todo y no funciona.'),
    (4, 'Muy útil en el día a día', 'Uso las notificaciones y el seguimiento de pasos todo el día. La batería alcanza justo.'),
    (5, 'Calidad premium', 'Materiales de muy buena calidad y una pantalla excelente. Vale lo que cuesta.'),
    (2, 'Mediciones poco confiables', 'Los pasos y las calorías no coinciden con la realidad. No confío en los datos.'),
    (5, 'Buenísimo para caminar', 'Lo uso para caminar todos los días. El GPS y los pasos son muy precisos.'),
    (4, 'Cumple lo prometido', 'Funciona bien y la batería dura lo que dicen. La aplicación tiene muchas opciones.'),
    (1, 'Pantalla se apaga sola', 'La pantalla se apaga sola y hay que reiniciarlo. Muy mala experiencia.'),
]

try:
    crudo = descargar_resenas(SKUS)
    resenas = (crudo.drop_duplicates(subset="Id")
                    .rename(columns={"Rating": "nota", "Title": "titulo", "ReviewText": "texto"})
                    [["nota", "titulo", "texto"]])
    if len(resenas) < 30:
        raise ValueError("Muy pocas reseñas")
    FUENTE = "API de Samsung (Bazaarvoice)"
except Exception as e:
    print("No se pudo usar la API:", type(e).__name__, "| Uso las reseñas de respaldo (ficticias).")
    resenas = pd.DataFrame(RESPALDO, columns=["nota", "titulo", "texto"])
    FUENTE = "respaldo ficticio"

resenas = resenas.dropna(subset=["texto"]).reset_index(drop=True)
resenas["titulo"] = resenas["titulo"].fillna("")
print(f"\nFuente: {FUENTE} | Reseñas: {len(resenas)}")
resenas.head()

**Qué deberías ver:** si la API funciona, varias páginas descargadas y algunos cientos de reseñas reales. Si no, el mensaje de respaldo y 60 reseñas.

**Por qué `drop_duplicates(subset="Id")`:** Bazaarvoice agrupa las reseñas por familia de producto, así que el mismo reloj en distintos colores puede devolver las mismas reseñas. Cada reseña tiene un `Id` único, y así quitamos las repetidas.

## Paso 2.3. Explorar

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
resenas["nota"].value_counts().sort_index().plot(kind="bar", ax=ejes[0], color="#1428A0")
ejes[0].set_title("Reseñas por nota"); ejes[0].set_xlabel("Nota (1 a 5)")
resenas["texto"].str.split().str.len().plot(kind="hist", bins=20, ax=ejes[1], color="#FF6B35")
ejes[1].set_title("Largo de las reseñas"); ejes[1].set_xlabel("Palabras")
plt.tight_layout(); plt.show()

print("Ejemplo de reseña con nota 1:\n", resenas.loc[resenas["nota"] == 1, "texto"].iloc[0])

**Qué deberías ver:** casi siempre hay muchas más reseñas con nota 5 que con nota 1. Es un **desbalance**, igual que en el caso de fuga de clientes de la Clase 6, y lo tendremos en cuenta en la Parte 6.

### Ejercicio 2
Guarda en `prop_positivas` la **proporción** de reseñas con nota 4 o 5 (un número entre 0 y 1).

In [ ]:
prop_positivas = None  # TU CÓDIGO AQUÍ
print(prop_positivas)

In [ ]:
revisar(lambda: np.isclose(prop_positivas, (resenas["nota"] >= 4).mean()))

---
# Parte 3. Expresiones regulares

Repaso rápido de la tabla de ayer:

| Símbolo | Significa | Ejemplo |
|---|---|---|
| `\d` | Un dígito | `\d{4}` encuentra `2026` |
| `\w` | Letra, dígito o `_` | `\w+` encuentra una palabra |
| `\s` | Un espacio | |
| `+` / `*` / `?` | Una o más / cero o más / opcional | |
| `[...]` | Uno de estos caracteres | `[kK]` |
| `(?:...)` | Agrupa sin guardar aparte | `(?:\.\d{3})*` |
| `\` | Usa el símbolo literal | `\.` es un punto |

`re.findall(patrón, texto)` extrae y `re.sub(patrón, reemplazo, texto)` reemplaza.

### Ejercicio 3 (el que quedó pendiente ayer)
Escribe `patron_porcentaje` para encontrar `45%`, `3,5%` y `100%`. Pista: dígitos, opcionalmente una coma con más dígitos, y `%`.

In [ ]:
texto_porcentajes = "La aprobación subió a 45% y la inflación anual fue de 3,5%. La meta es llegar al 100% de cobertura."

patron_porcentaje = None  # TU CÓDIGO AQUÍ, por ejemplo r"..."
print(re.findall(patron_porcentaje, texto_porcentajes) if patron_porcentaje else None)

In [ ]:
revisar(lambda: re.findall(patron_porcentaje, texto_porcentajes) == ["45%", "3,5%", "100%"])

## Paso 3.1. Limpiar texto de usuarios

Las reseñas, los comentarios y las publicaciones traen ruido típico: enlaces, menciones, signos repetidos y espacios de más.

**Qué vamos a hacer:** escribir una función `limpiar` con varias reglas de `re.sub` y probarla.

In [ ]:
def limpiar(texto):
    texto = re.sub(r"https?://\S+", " ", texto)        # enlaces: http o https, y todo hasta el siguiente espacio
    texto = re.sub(r"@\w+", " ", texto)                # menciones como @samsungchile
    texto = re.sub(r"([!?.])\1+", r"\1", texto)         # signos repetidos: "!!!" pasa a "!"
    texto = re.sub(r"\s+", " ", texto)                  # varios espacios o saltos de línea pasan a uno
    return texto.strip()

ejemplo_sucio = "Llegó rápido!!!   Lo compré en https://www.samsung.com/cl y @samsungchile respondió altiro...\n\nMuy bueno"
print("Antes:  ", repr(ejemplo_sucio))
print("Después:", repr(limpiar(ejemplo_sucio)))

**Qué deberías ver:** sin el enlace, sin la mención, con un solo signo de exclamación y sin espacios de más.

**Cómo leer `([!?.])\1+`:** los paréntesis **guardan** el signo encontrado, y `\1` significa "el mismo signo que guardé". Entonces encuentra un signo seguido de una o más repeticiones de sí mismo.

### Ejercicio 4
Escribe `patron_hashtag` para extraer los hashtags (como `#GalaxyWatch`) del texto de la celda.

In [ ]:
texto_hashtags = "Feliz con mi nuevo reloj #GalaxyWatch #Salud. Lo uso para correr todos los días #Running"

patron_hashtag = None  # TU CÓDIGO AQUÍ
print(re.findall(patron_hashtag, texto_hashtags) if patron_hashtag else None)

In [ ]:
revisar(lambda: re.findall(patron_hashtag, texto_hashtags) == ["#GalaxyWatch", "#Salud", "#Running"])

Aplicamos la limpieza a todas las reseñas. Juntamos el título y el texto, porque el título suele resumir la opinión.

In [ ]:
resenas["completo"] = (resenas["titulo"] + ". " + resenas["texto"]).apply(limpiar)
resenas[["nota", "completo"]].head()

---
# Parte 4. Preprocesamiento

Lo hacemos paso a paso con una sola frase, para ver qué hace cada paso.

In [ ]:
frase = "Los Bancos Centrales subieron las tasas de interés, pero la economía chilena creció 2,5% en 2026."

paso1 = frase.lower()                                   # 1. minúsculas
paso2 = re.sub(r"[^a-záéíóúñü\s]", " ", paso1)          # 2. todo lo que no sea letra pasa a espacio
tokens = paso2.split()                                  # 3. tokenizar

print("1. Minúsculas: ", paso1)
print("2. Solo letras:", paso2)
print("3. Tokens:     ", tokens, f"({len(tokens)} tokens)")

**Qué deberías ver:** la frase en minúsculas, luego sin comas ni números, y luego como una lista de 14 tokens.

`[^...]` significa "cualquier carácter que **no** esté en la lista". Ojo: aquí quitamos los números porque para hablar de temas no aportan; en otros problemas (reclamos con montos, por ejemplo) sí importan.

## Paso 4.1. Stopwords, y una decisión importante

> **Definición. Stopwords:** palabras muy frecuentes que casi no aportan significado sobre el tema, como "el", "de", "que" o "pero". NLTK trae 313 en español.

In [ ]:
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("spanish"))
print("Sin stopwords:", [t for t in tokens if t not in STOPWORDS])
print("\n¿'no' es stopword?", "no" in STOPWORDS, "| ¿'muy'?", "muy" in STOPWORDS, "| ¿'sin'?", "sin" in STOPWORDS)

**Qué deberías ver:** desaparecen "los", "las", "de", "pero", "la" y "en". Y **"no", "muy" y "sin" están en la lista de stopwords**.

Eso es un problema para las reseñas: "funciona" y "**no** funciona" quedarían iguales. **La lista de stopwords depende del problema.** Para hablar de temas, "no" sobra; para saber si una opinión es positiva o negativa, es de las palabras más importantes. Por eso creamos una segunda lista que conserva las negaciones e intensificadores.

In [ ]:
CONSERVAR = {"no", "sin", "nada", "nunca", "muy", "más", "poco"}
STOPWORDS_OPINION = STOPWORDS - CONSERVAR
print("Palabras que conservamos:", sorted(CONSERVAR))

## Paso 4.2. Stemming y lematización

| | Stemming | Lematización |
|---|---|---|
| Qué hace | Corta la palabra para dejar su raíz | Busca la forma de diccionario |
| "subieron" | `sub` | `subir` |
| "chilena" | `chilen` | `chileno` |
| Ventaja | Muy rápido | Siempre es una palabra real |
| Herramienta | NLTK `SnowballStemmer` | spaCy |

In [ ]:
from nltk.stem import SnowballStemmer
import spacy

stemmer = SnowballStemmer("spanish")
nlp = spacy.load("es_core_news_sm")

palabras = [t for t in tokens if t not in STOPWORDS]
pd.DataFrame({"palabra": palabras,
              "stemming": [stemmer.stem(t) for t in palabras],
              "lematización": [t.lemma_ for t in nlp(" ".join(palabras))]})

**Qué deberías ver:** el stemming deja raíces como `econom`; la lematización, palabras completas como `economía`.

## Paso 4.3. Todo junto

**Qué vamos a hacer:** una función `preprocesar` con dos opciones: el método de normalización y la lista de stopwords.

In [ ]:
def preprocesar(texto, metodo="stemming", stopwords_usar=STOPWORDS):
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto.lower())        # minúsculas y solo letras
    tokens = [t for t in texto.split()
              if t not in stopwords_usar and (len(t) > 2 or t == "no")]   # sin stopwords ni palabras muy cortas (salvo "no")
    if metodo == "stemming":
        tokens = [stemmer.stem(t) for t in tokens]
    elif metodo == "lematizacion":
        tokens = [t.lemma_ for t in nlp(" ".join(tokens))]
    return " ".join(tokens)

opinion = "No funciona bien, la batería no dura nada y es muy cara."
print("Para temas:   ", preprocesar(opinion))
print("Para opinión: ", preprocesar(opinion, stopwords_usar=STOPWORDS_OPINION))

**Qué deberías ver:** con la lista normal, la opinión pierde "no", "nada" y "muy", y parece neutral. Con la lista para opiniones, se conserva lo negativo.

### Ejercicio 5
Usa `preprocesar` (con stemming y la lista normal) sobre `otra_frase` y guarda el resultado en `frase_procesada`.

In [ ]:
otra_frase = "Las jugadoras chilenas ganaron el campeonato y celebraron con la hinchada en el estadio."

frase_procesada = None  # TU CÓDIGO AQUÍ
print(frase_procesada)

In [ ]:
revisar(lambda: frase_procesada == preprocesar(otra_frase))

Preprocesamos las reseñas de las dos formas, porque las usaremos para dos preguntas distintas en la Parte 6.

In [ ]:
resenas["para_temas"] = resenas["completo"].apply(preprocesar)
resenas["para_opinion"] = resenas["completo"].apply(lambda t: preprocesar(t, stopwords_usar=STOPWORDS_OPINION))
resenas[["completo", "para_temas", "para_opinion"]].head(3)

---
# Parte 5. Del texto a números: TF-IDF

> **Bolsa de palabras:** cada documento se representa como el conteo de cada palabra, sin importar el orden.
>
> **TF-IDF:** multiplica la frecuencia de la palabra en el documento (TF) por qué tan rara es en el corpus (IDF). Premia las palabras características de un documento.

scikit-learn calcula:

$$\text{IDF} = \ln\left(\frac{1 + n}{1 + df}\right) + 1$$

donde **n** es el número de documentos y **df** el número de documentos que contienen la palabra.

## Paso 5.1. A mano

Corpus de 3 documentos: "gol partido gol", "partido tenis", "tasa interés tasa".

- **gol** aparece en 1 documento: IDF = ln(4/2) + 1 = **1,693**. En el documento 1 aparece 2 veces: TF-IDF = 2 × 1,693 = **3,386**.
- **partido** aparece en 2 documentos: IDF = ln(4/3) + 1 = **1,288**. En el documento 1 aparece 1 vez: TF-IDF = **1,288**.

"Partido" pesa menos porque es menos característica.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

mini_corpus = ["gol partido gol", "partido tenis", "tasa interés tasa"]
indices = ["doc 1", "doc 2", "doc 3"]

conteo = CountVectorizer()
print("Bolsa de palabras:")
display(pd.DataFrame(conteo.fit_transform(mini_corpus).toarray(), columns=conteo.get_feature_names_out(), index=indices))

sin_norma = TfidfVectorizer(norm=None)
print("TF-IDF sin normalizar (compáralo con el cálculo a mano):")
display(pd.DataFrame(sin_norma.fit_transform(mini_corpus).toarray().round(3), columns=sin_norma.get_feature_names_out(), index=indices))

tfidf = TfidfVectorizer()
print("TF-IDF normalizado (cada fila tiene largo 1):")
display(pd.DataFrame(tfidf.fit_transform(mini_corpus).toarray().round(3), columns=tfidf.get_feature_names_out(), index=indices))

**Qué deberías ver:** en la segunda tabla, `gol = 3,386` y `partido = 1,288` en el documento 1, igual que a mano. La tercera tabla divide cada fila por su largo, para que los documentos largos no tengan ventaja.

### Ejercicio 6
Con un cuarto documento, `"partido gol tenis"`, hay n = 4 documentos y "partido" aparece en 3. Calcula **a mano** su IDF con `np.log` y guárdalo en `idf_partido`.

In [ ]:
idf_partido = None  # TU CÓDIGO AQUÍ
print(idf_partido)

In [ ]:
_v = TfidfVectorizer().fit(mini_corpus + ["partido gol tenis"])
revisar(lambda: np.isclose(idf_partido, _v.idf_[list(_v.get_feature_names_out()).index("partido")]))

---
# Parte 6. Aplicaciones con las reseñas

Con las reseñas convertidas en números, respondemos dos preguntas que le importan a cualquier empresa:

1. **¿De qué hablan los clientes?** Agrupamiento con K-medias (no supervisado, Clases 7 a 10).
2. **¿La reseña es positiva o negativa?** Clasificación con regresión logística (supervisado, Clases 4 a 6).

## Paso 6.1. ¿De qué hablan? Agrupar por temas

**Qué vamos a hacer:** TF-IDF sobre el texto preprocesado "para temas", K-medias con k = 5, y para cada grupo: sus palabras clave, cuántas reseñas tiene y su nota promedio.

In [ ]:
from sklearn.cluster import KMeans

vect_temas = TfidfVectorizer(min_df=2)          # min_df=2: ignorar palabras que aparecen en una sola reseña
X_temas = vect_temas.fit_transform(resenas["para_temas"])
print("Matriz:", X_temas.shape, "(reseñas × palabras)")

K = 5
km = KMeans(n_clusters=K, n_init=20, random_state=42).fit(X_temas)
resenas["tema"] = km.labels_

terminos = vect_temas.get_feature_names_out()
resumen_temas = pd.DataFrame({
    "palabras_clave": [", ".join(terminos[np.argsort(c)[::-1][:6]]) for c in km.cluster_centers_],
    "reseñas": np.bincount(km.labels_, minlength=K),
    "nota_promedio": resenas.groupby("tema")["nota"].mean().round(2).reindex(range(K)).values,
})
resumen_temas.sort_values("nota_promedio")

**Qué deberías ver:** grupos con palabras clave que se pueden nombrar, como batería, pantalla, conexión con el celular, sueño o salud. La nota promedio muestra **qué temas generan más molestia**: es exactamente lo que un equipo de producto quiere saber.

Leamos dos reseñas del tema con peor nota, para confirmar que el nombre tiene sentido:

In [ ]:
tema_peor = resenas.groupby("tema")["nota"].mean().idxmin()
for t in resenas.loc[resenas["tema"] == tema_peor, "completo"].head(2):
    print("-", t)

### Ejercicio 7
Guarda en `tema_mejor` el número del tema con la **nota promedio más alta**, y en `nombre_tema_mejor` un nombre corto que tú le pongas mirando sus palabras clave (este no se verifica).

In [ ]:
tema_mejor = None         # TU CÓDIGO AQUÍ
nombre_tema_mejor = ""    # TU NOMBRE AQUÍ
print(tema_mejor, nombre_tema_mejor)

In [ ]:
revisar(lambda: tema_mejor == resenas.groupby("tema")["nota"].mean().idxmax())

## Paso 6.2. ¿Positiva o negativa? Clasificar opiniones

**Qué vamos a hacer:**

1. Crear la variable objetivo: **positiva** (nota 4 o 5) o **negativa** (nota 1 o 2). Dejamos fuera las de nota 3, que son ambiguas.
2. Usar el texto preprocesado **para opinión** (el que conserva "no").
3. Armar un `Pipeline` con TF-IDF y regresión logística con `class_weight="balanced"`, por el desbalance.
4. Evaluar con validación cruzada y mirar **precision, recall y F1** de la clase negativa, que es la que más le importa a la empresa (Clase 7).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report

opiniones = resenas[resenas["nota"] != 3].copy()
opiniones["sentimiento"] = np.where(opiniones["nota"] >= 4, "positiva", "negativa")
print(opiniones["sentimiento"].value_counts())

clasificador = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("modelo", LogisticRegression(class_weight="balanced", max_iter=1000)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
prediccion = cross_val_predict(clasificador, opiniones["para_opinion"], opiniones["sentimiento"], cv=cv)
print(classification_report(opiniones["sentimiento"], prediccion, digits=3))

**Qué deberías ver:** un reporte con precision, recall y F1 para cada clase. Normalmente la clase positiva tiene mejores números porque hay más ejemplos. **Miren el recall de la clase negativa:** de las reseñas negativas reales, ¿cuántas detecta el modelo?

`cross_val_predict` entrega, para cada reseña, la predicción de un modelo que **no la vio** al entrenar. Así el reporte es honesto.

### ¿Qué palabras aprendió el modelo?

In [ ]:
clasificador.fit(opiniones["para_opinion"], opiniones["sentimiento"])
coef = pd.Series(clasificador.named_steps["modelo"].coef_[0], index=clasificador.named_steps["tfidf"].get_feature_names_out())
# La clase "positiva" es la segunda en orden alfabético: coeficiente positivo empuja hacia "positiva"
pd.concat([coef.nsmallest(8), coef.nlargest(8)]).plot(kind="barh", figsize=(7, 5), color=["#D9534F"] * 8 + ["#1E9E73"] * 8,
                                                      title="Palabras que empujan hacia negativa (rojo) o positiva (verde)")
plt.show()

**Qué deberías ver:** **"no"** entre las palabras que más empujan hacia negativa. Si hubiéramos usado la lista normal de stopwords, esa palabra no existiría para el modelo: la decisión del Paso 4.1 importa.

Probemos con opiniones nuevas:

In [ ]:
nuevas = ["La batería dura dos días y la pantalla es excelente",
          "No sincroniza con el celular y el servicio técnico no responde",
          "Muy cómodo, lo uso para dormir y medir el sueño"]
for texto, pred in zip(nuevas, clasificador.predict([preprocesar(limpiar(t), stopwords_usar=STOPWORDS_OPINION) for t in nuevas])):
    print(f"{pred:9s} | {texto}")

### Ejercicio 8
Del reporte de clasificación, guarda en `f1_negativa` el **F1 de la clase negativa** (un número entre 0 y 1). Pista: `f1_score(..., pos_label="negativa")` con `opiniones["sentimiento"]` y `prediccion`.

In [ ]:
from sklearn.metrics import f1_score

f1_negativa = None  # TU CÓDIGO AQUÍ
print(f1_negativa)

In [ ]:
revisar(lambda: np.isclose(f1_negativa, f1_score(opiniones["sentimiento"], prediccion, pos_label="negativa")))

---
# Cierre

| Tema | Lo esencial |
|---|---|
| Scraping | Si lo que ves en el navegador no está en `res.text`, la página es dinámica. El mejor camino es encontrar la API en la pestaña *Network*. |
| BeautifulSoup | `find` devuelve uno (o `None`); `find_all`, una lista. |
| APIs y JSON | `res.json()` entrega diccionarios y listas de Python. Pocas peticiones, con pausa. |
| Regex | `re.sub` para limpiar enlaces, menciones y signos repetidos. |
| Stopwords | La lista depende del problema: para opiniones, "no" es clave. |
| TF-IDF | Premia lo característico de cada documento. |
| Aplicaciones | Temas con K-medias y sentimiento con regresión logística, evaluada con F1 de la clase que importa. |

**Para el capstone:** si su proyecto usa texto, este notebook es una plantilla completa: obtener, limpiar, preprocesar, TF-IDF, agrupar y clasificar.

Guarden el notebook en `ejercicios_estudiantes` como `clase11_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1

In [ ]:
precios = [int(p.get_text(strip=True).replace("$", "").replace(".", "")) for p in sopa.find_all(class_="precio")]
print(precios)

### Ejercicio 2

In [ ]:
prop_positivas = (resenas["nota"] >= 4).mean()
print(round(prop_positivas, 3))

### Ejercicio 3

In [ ]:
patron_porcentaje = r"\d+(?:,\d+)?%"
print(re.findall(patron_porcentaje, texto_porcentajes))

### Ejercicio 4
Un `#` seguido de una o más letras, dígitos o guiones bajos.

In [ ]:
patron_hashtag = r"#\w+"
print(re.findall(patron_hashtag, texto_hashtags))

### Ejercicio 5

In [ ]:
frase_procesada = preprocesar(otra_frase)
print(frase_procesada)

### Ejercicio 6

In [ ]:
idf_partido = np.log((1 + 4) / (1 + 3)) + 1
print(round(idf_partido, 3))

### Ejercicio 7

In [ ]:
tema_mejor = resenas.groupby("tema")["nota"].mean().idxmax()
print(tema_mejor, "|", resumen_temas.loc[tema_mejor, "palabras_clave"])

### Ejercicio 8

In [ ]:
f1_negativa = f1_score(opiniones["sentimiento"], prediccion, pos_label="negativa")
print(round(f1_negativa, 3))